## Generating synthetic QA dataset based on several key papers in GenAI

In [1]:
import json
from pathlib import Path

import openai
from pypdf import PdfReader
from tqdm.auto import tqdm

c:\Users\mikes\Documents\STUDY\LLM-zoomcamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------- CONFIG --------
PARENT_DIR = Path("C:/Users/mikes/Documents/STUDY/llm_fine_tuning_data")
INPUT_DIR = PARENT_DIR / "pdf"      # folder with .pdf or .txt files
OUTPUT_FILE = PARENT_DIR / "raw_qa_1.jsonl"
QUESTIONS_PER_CHUNK = 6
CHUNK_SIZE = 2000  # characters
MODEL_NAME = "gpt-3.5-turbo"
TEMPERATURE = 0.2
# ------------------------


In [3]:
client = openai.OpenAI()

def chunk_text(text, size):
    for i in range(0, len(text), size):
        yield text[i:i+size]

def remove_references(text):
    """Remove references/bibliography section from text"""
    import re
    
    # Common patterns for references sections
    patterns = [
        r'\n\s*REFERENCES\s*\n',
        r'\n\s*References\s*\n',
        r'\n\s*BIBLIOGRAPHY\s*\n',
        r'\n\s*Bibliography\s*\n',
        r'\n\s*\d+\s*REFERENCES\s*\n',  # numbered sections like "5 REFERENCES"
        r'\n\s*\[\d+\]\s*References\s*\n',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            text = text[:match.start()]
            break
    
    return text

def read_pdf(path):
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    text = "\n".join(pages)
    return remove_references(text)

def read_txt(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    return remove_references(text)

def generate_qa(text_chunk):
    prompt = f"""
You are creating a machine learning QA dataset.

From the text below, generate {QUESTIONS_PER_CHUNK} technical questions and concise answers.
Focus on definitions, assumptions, limitations, and method comparisons.

Return JSON with a list under key "qa", where each item has "question" and "answer".
Text:
{text_chunk}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": "You create concise QA pairs for machine learning study."},
            {"role": "user", "content": prompt},
        ],
    )

    content = response.choices[0].message.content.strip()

    #print(content)

    try:
        parsed = json.loads(content)
        if isinstance(parsed, dict) and "qa" in parsed:
            parsed = parsed["qa"]
        if not isinstance(parsed, list):
            raise ValueError("Response is not a list")

        cleaned = []
        for item in parsed:
            q = (item.get("question") or "").strip()
            a = (item.get("answer") or "").strip()
            if q and a:
                cleaned.append({"question": q, "answer": a})

        return cleaned[:QUESTIONS_PER_CHUNK]
    except Exception:
        return []


In [26]:

files = list(Path(INPUT_DIR).glob("*.pdf")) + list(Path(INPUT_DIR).glob("*.txt"))
files = sorted(files)

In [28]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for file in tqdm(files, desc="Processing files"):
        if file.suffix.lower() == ".pdf":
            text = read_pdf(file)
        else:
            text = read_txt(file)

        chunks = list(chunk_text(text, CHUNK_SIZE))

        print(len(chunks))

Processing files:  10%|█         | 1/10 [00:00<00:06,  1.40it/s]

32


Processing files:  20%|██        | 2/10 [00:01<00:06,  1.23it/s]

26


Processing files:  30%|███       | 3/10 [00:01<00:04,  1.66it/s]

...

22


Processing files:  90%|█████████ | 9/10 [00:07<00:00,  1.14it/s]

26


Processing files: 100%|██████████| 10/10 [00:09<00:00,  1.06it/s]

60


In [29]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for file in tqdm(files, desc="Processing files"):
        if file.suffix.lower() == ".pdf":
            text = read_pdf(file)
        else:
            text = read_txt(file)

        chunks = list(chunk_text(text, CHUNK_SIZE))
        ind =0
        for chunk in tqdm(chunks, desc=f"  {file.name}", leave=False):
            #print(f"ind_chunk: {ind}")
            qa_pairs = generate_qa(chunk)
            for qa in qa_pairs:
                out.write(json.dumps(qa, ensure_ascii=False) + "\n")
            ind += 1

print("✅ Saved raw QA pairs to", OUTPUT_FILE)

Processing files: 100%|██████████| 10/10 [16:36<00:00, 99.62s/it]

✅ Saved raw QA pairs to C:\Users\mikes\Documents\STUDY\llm_fine_tuning_data\raw_qa_1.jsonl


In [30]:
d = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            d.append(json.loads(line))

print(f"Loaded data from {OUTPUT_FILE}, total items: {len(d)}")
print(f"Example item:\n{d[0]}")


Loaded data from C:\Users\mikes\Documents\STUDY\llm_fine_tuning_data\raw_qa_1.jsonl, total items: 294
Example item:
{'question': "What is the main problem addressed in the paper 'Neural Machine Translation of Rare Words with Subword Units'?", 'answer': 'The paper addresses the issue of translating rare and unknown words in neural machine translation models, which typically operate with a fixed vocabulary.'}


In [4]:
OUTPUT_FILE = PARENT_DIR / "raw_qa.jsonl"

OUTPUT_FILE_1 = PARENT_DIR / "raw_qa_1.jsonl"

merge=[]

with open(OUTPUT_FILE, "r", encoding="utf-8") as jsonout:
    
    for l in jsonout:
        merge.append(l)

with open(OUTPUT_FILE_1, "r", encoding="utf-8") as jsonout:
    
    for l in jsonout:
        merge.append(l)

OUTPUT_FILE_MERGED = PARENT_DIR / "raw_qa_merged.jsonl"
with open(OUTPUT_FILE_MERGED, "w", encoding="utf-8") as out:
    for l in merge:
        out.write(l)

In [5]:
print(f"Merged dataset length: {len(merge)} items")

Merged dataset length: 510 items
